# Compare Base Round Robin, ARRTQ, and modified ARRTQ

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))  # add parent directory to search path

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from core.process import Process
from core.scheduler_arrtq import arrtq
from core.scheduler_rr import round_robin
from core.scheduler_modified_ARRTQ import arrtq_balanced
from core.process_generator import generate_processes
import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import copy

In [13]:
context_switch_time = 1
sizes = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
seed_base = 5  # to ensure different random sequences per dataset

# Results
summary = []

for i, size in enumerate(sizes):
    print(f"Running simulation for {size} processes...")

    processes = generate_processes(size, seed=seed_base + i)
   
    arrtq_metrics = arrtq(copy.deepcopy(processes), context_switch_time)
    arrtq_balanced_metrics = arrtq_balanced(copy.deepcopy(processes), context_switch_time, c=0.9)

    summary.extend([
    {
        "Processes": size,
        "Algorithm": "ARRTQ",
        "Avg_TAT": arrtq_metrics.get("average_turnaround_time", np.nan),
        "Avg_WT": arrtq_metrics.get("average_waiting_time", np.nan),
        "Avg_FRT": arrtq_metrics.get("average_first_response_time", np.nan),
        "Throughput": arrtq_metrics.get("throughput", np.nan),
        "CPU_Util": arrtq_metrics.get("cpu_utilization", np.nan),
        "Context_Switches": arrtq_metrics.get("context_switches", np.nan),
    },
    {
        "Processes": size,
        "Algorithm": "Balanced ARRTQ",
        "Avg_TAT": arrtq_balanced_metrics.get("average_turnaround_time", np.nan),
        "Avg_WT": arrtq_balanced_metrics.get("average_waiting_time", np.nan),
        "Avg_FRT": arrtq_balanced_metrics.get("average_first_response_time", np.nan),
        "Throughput": arrtq_balanced_metrics.get("throughput", np.nan),
        "CPU_Util": arrtq_balanced_metrics.get("cpu_utilization", np.nan),
        "Context_Switches": arrtq_balanced_metrics.get("context_switches", np.nan),
    },
])


# DataFrame
summary_df = pd.DataFrame(summary)
display(summary_df)


Running simulation for 1000 processes...
Running simulation for 2000 processes...
Running simulation for 3000 processes...
Running simulation for 4000 processes...
Running simulation for 5000 processes...
Running simulation for 6000 processes...
Running simulation for 7000 processes...
Running simulation for 8000 processes...
Running simulation for 9000 processes...
Running simulation for 10000 processes...


,Processes,Algorithm,Avg_TAT,Avg_WT,Avg_FRT,Throughput,CPU_Util,Context_Switches
0,1000,ARRTQ,5626.564495,5615.427495,4284.699806,0.075409,99.984918,2124
1,1000,Balanced ARRTQ,5831.567096,5820.430096,3295.039785,0.075672,99.984866,2078
2,2000,ARRTQ,10856.480784,10845.797284,8133.664041,0.078110,99.984378,4238
3,2000,Balanced ARRTQ,11070.963956,11060.280456,6195.667072,0.078638,99.984272,4066
4,3000,ARRTQ,16720.864567,16710.074901,12580.806389,0.077401,99.997420,6390
5,3000,Balanced ARRTQ,17147.550904,17136.761237,9475.531057,0.077740,99.997409,6221
6,4000,ARRTQ,23033.870923,23022.823423,17148.884990,0.075839,100.000000,8553
7,4000,Balanced ARRTQ,23476.669921,23465.622421,13118.377467,0.076172,100.000000,8323
8,5000,ARRTQ,28493.103113,28482.020113,21785.886759,0.075571,99.998489,10748
9,5000,Balanced ARRTQ,28992.987476,28981.904476,16270.200255,0.076128,99.998477,10264


In [14]:
def plot_metric(metric_name, y_label, color1='royalblue', color2='seagreen'):
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=summary_df[summary_df["Algorithm"] == "ARRTQ"]["Processes"],
        y=summary_df[summary_df["Algorithm"] == "ARRTQ"][metric_name],
        name='ARRTQ',
        marker_color=color1
    ))

    fig.add_trace(go.Bar(
        x=summary_df[summary_df["Algorithm"] == "Balanced ARRTQ"]["Processes"],
        y=summary_df[summary_df["Algorithm"] == "Balanced ARRTQ"][metric_name],
        name='Balanced ARRTQ',
        marker_color=color2
    ))

    fig.update_layout(
        title=f'{y_label} Comparison: ARRTQ vs Balanced ARRTQ',
        xaxis_title='Number of Processes',
        yaxis_title=y_label,
        barmode='group',
        template='plotly_white'
    )
    fig.show()

# Plot 3 key metrics
plot_metric("Avg_FRT", "Average First Response Time")
plot_metric("Avg_TAT", "Average Turnaround Time")
plot_metric("Avg_WT", "Average Waiting Time")
# --- Plot additional performance metrics ---

plot_metric("CPU_Util", "CPU Utilization (%)", color1='royalblue', color2='seagreen')

plot_metric("Context_Switches", "Number of Context Switches", color1='royalblue', color2='seagreen')

plot_metric("Throughput", "Throughput (Processes / Time Unit)", color1='royalblue', color2='seagreen')
